# **Machine Learning Project PART 2 - ATU Winter 2024**  
**Author**: Lais Coletta Pereira  
**Lecturer**: Brian McGinley  

---

## Step 1 – Data Pre-processing and management:
The provided uncompressed wav files are large and there is lots of the recordings whether there
are no calls present.
The first task will be to build a dataset from the annotated data. I’ve provided a jupyter notebook
that will assist in this regard.
My advice would be to extract a spectrogram for each call. Each spectrogram should be the
same size so there will be some pre-pre-processing to find what is the longest call (in time) and
the broadest in frequency. This will serve as the baseline for the largest spectrogram. Extract
and save (with the metadata)a spectrogram for each call (you can calculate the central time of
each call from the metadata). Note, don’t save as images but as a raw 2d array of numbers.
You can also create spectrograms for the extra class “no-call” where you build a set of
spectrograms from times when there is no annotated call. You may assume that any
unannotated region has no call in it. Take care to ensure that the extracted“no-call”
spectrograms are from the same frequency region as the call spectrograms.
I would recommend having a single jupyter notebook file that does all this preprocessing.

Data Pre-Processing Steps Plan:
Organize and load audio and metadata files for initial inspection.
Calculate the maximum time and frequency dimensions of the spectrogram.
Extract spectrograms from the annotated calls as well as the unannotated "no-call" regions.
Zero-pad spectrograms so they are consistent in size.
Save the spectrograms as raw 2D NumPy arrays, and optionally store metadata (such as labels and file names) in CSV files.
Validate that all spectrograms are consistently sized and properly labeled.


In [ ]:
import pandas as pd

# Example of reading annotations
annotations_file = 'data/annotations/call_1.csv'
annotations = pd.read_csv(annotations_file)
print(annotations.head())


In [ ]:
import librosa

# Function to load audio and get its duration
def get_audio_duration(file_path):
    y, sr = librosa.load(file_path, sr=None)
    return librosa.get_duration(y=y, sr=sr), sr  # duration in seconds and sampling rate

audio_file = 'data/raw_audio/seal_call_1.wav'
duration, sr = get_audio_duration(audio_file)
print(f"Duration of audio: {duration} seconds, Sample Rate: {sr}")


In [ ]:
# Generating a spectrogram for frequency range
import matplotlib.pyplot as plt

def generate_spectrogram(file_path):
    y, sr = librosa.load(file_path, sr=None)
    D = librosa.amplitude_to_db(librosa.stft(y), ref=np.max)
    
    plt.figure(figsize=(10, 6))
    librosa.display.specshow(D, x_axis='time', y_axis='log', sr=sr)
    plt.title('Spectrogram')
    plt.colorbar(format='%+2.0f dB')
    plt.show()

generate_spectrogram(audio_file)


In [ ]:
import numpy as np

def extract_spectrogram_for_segment(file_path, start_time, end_time, target_duration, target_freq_bins):
    y, sr = librosa.load(file_path, sr=None, offset=start_time, duration=(end_time-start_time))
    
    # Generate a spectrogram (STFT or MelSpectrogram)
    S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=target_freq_bins)
    
    # Convert to log scale (decibel)
    log_S = librosa.power_to_db(S, ref=np.max)
    
    # Zero-padding to ensure all spectrograms are the same size
    if log_S.shape[1] < target_duration:
        pad_width = target_duration - log_S.shape[1]
        log_S = np.pad(log_S, ((0, 0), (0, pad_width)), mode='constant')
    
    return log_S

# Example call for an audio segment
start_time = 2.3
end_time = 5.7
target_duration = 300  # Target columns for consistent time span (e.g., 300 time steps)
target_freq_bins = 128  # Target frequency bins

spectrogram = extract_spectrogram_for_segment('data/raw_audio/seal_call_1.wav', start_time, end_time, target_duration, target_freq_bins)


In [ ]:
import random

def extract_no_call_segment(file_path, call_duration, target_duration, target_freq_bins):
    # Select random start and end times in non-call regions
    start_time = random.uniform(0, call_duration - 10)  # 10 second padding as an example
    end_time = start_time + 5  # Random 5 second segment as an example
    
    return extract_spectrogram_for_segment(file_path, start_time, end_time, target_duration, target_freq_bins)

# Example for extracting a "no-call" segment
no_call_spectrogram = extract_no_call_segment('data/raw_audio/seal_call_1.wav', 300, target_duration, target_freq_bins)
